# 02 — Processor

Processes downloaded PDFs: detects whether pages are searchable or scanned, extracts text directly or via OCR, and produces searchable PDFs + text files.

**What this notebook does:**
1. Scans the download folder for PDF files
2. Checks each PDF for extractable text using PyMuPDF
3. Searchable PDFs → extract text directly
4. Scanned/image PDFs → OCR with Tesseract, produce `_searchable.pdf`
5. Saves `.txt` files for each volume
6. Tracks progress in `processing_manifest.json`

In [15]:
import os
import json
import logging
from pathlib import Path
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
from multiprocessing import cpu_count

import fitz  # PyMuPDF
from dotenv import load_dotenv
from tqdm.notebook import tqdm

## Configuration

In [16]:
load_dotenv()

DOWNLOAD_PATH = Path(os.getenv("DOWNLOAD_PATH", "./downloads"))
MANIFEST_PATH = DOWNLOAD_PATH / "processing_manifest.json"

# Minimum characters per page to consider it "searchable"
MIN_TEXT_CHARS = 50

# Number of pages to sample when detecting if a PDF is scanned
SAMPLE_PAGES = 5

# OCR settings
OCR_DPI = 150
OCR_BATCH_SIZE = 50
OCR_WORKERS = max(1, cpu_count() - 1)

# Searchable PDF generation — off by default (notebook 04 reads originals via pdfplumber)
GENERATE_SEARCHABLE_PDF = False

print(f"Download path: {DOWNLOAD_PATH.resolve()}")
print(f"Manifest: {MANIFEST_PATH}")
print(f"OCR: DPI={OCR_DPI}, batch_size={OCR_BATCH_SIZE}, workers={OCR_WORKERS}")
print(f"Searchable PDF generation: {GENERATE_SEARCHABLE_PDF}")

Download path: C:\Users\Gamers\GitHub\SC-Decisions\downloads
Manifest: downloads\processing_manifest.json
OCR: DPI=150, batch_size=50, workers=11
Searchable PDF generation: False


## Logging Setup

In [17]:
logger = logging.getLogger("processor")
logger.setLevel(logging.INFO)
logger.handlers.clear()

file_handler = logging.FileHandler("processing.log", encoding="utf-8")
file_handler.setFormatter(logging.Formatter("%(asctime)s [%(levelname)s] %(message)s"))
logger.addHandler(file_handler)

console_handler = logging.StreamHandler()
console_handler.setFormatter(logging.Formatter("[%(levelname)s] %(message)s"))
logger.addHandler(console_handler)

logger.info("Processor initialized")

[INFO] Processor initialized


## Load Processing Manifest

The manifest tracks which files have already been processed, allowing safe re-runs.

In [18]:
def load_manifest():
    """Load the processing manifest from disk."""
    if MANIFEST_PATH.exists():
        with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
            return json.load(f)
    return {}

def save_manifest(manifest):
    """Save the processing manifest to disk."""
    with open(MANIFEST_PATH, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2, default=str)

manifest = load_manifest()
print(f"Manifest loaded: {len(manifest)} entries")

Manifest loaded: 860 entries


## Scan for PDFs

In [19]:
# Find all original PDFs (exclude _searchable.pdf files)
pdf_files = sorted([
    f for f in DOWNLOAD_PATH.glob("*.pdf")
    if not f.stem.endswith("_searchable")
])

print(f"Found {len(pdf_files)} PDF files to process")

Found 860 PDF files to process


## Helper Functions

In [20]:
def is_searchable(pdf_path, sample_pages=SAMPLE_PAGES, min_chars=MIN_TEXT_CHARS):
    """
    Check if a PDF has extractable text by sampling pages.
    Returns True if most sampled pages have sufficient text.
    """
    doc = fitz.open(pdf_path)
    total_pages = len(doc)
    pages_to_check = min(sample_pages, total_pages)

    # Sample evenly spaced pages
    if total_pages <= sample_pages:
        indices = range(total_pages)
    else:
        step = total_pages / sample_pages
        indices = [int(i * step) for i in range(sample_pages)]

    text_pages = 0
    for idx in indices:
        page = doc[idx]
        text = page.get_text().strip()
        if len(text) >= min_chars:
            text_pages += 1

    doc.close()

    # Consider searchable if majority of sampled pages have text
    return text_pages > pages_to_check / 2


def extract_text_pymupdf(pdf_path):
    """Extract text from all pages of a searchable PDF using PyMuPDF."""
    doc = fitz.open(pdf_path)
    all_text = []
    for page_num in range(len(doc)):
        page = doc[page_num]
        text = page.get_text()
        if text.strip():
            all_text.append(f"--- Page {page_num + 1} ---\n{text}")
    doc.close()
    return "\n\n".join(all_text)


def _ocr_single_page(page_num, image):
    """OCR a single page image. Runs in a thread — Tesseract is an external
    process so it releases the GIL; no pickle overhead unlike multiprocessing."""
    import pytesseract
    return page_num, pytesseract.image_to_string(image)


def ocr_pdf(pdf_path, output_searchable_path, manifest, manifest_key):
    """
    OCR a scanned PDF with batched conversion, threaded Tesseract, and
    page-level checkpointing for resume.

    Key performance choices vs. the previous version:
    - ThreadPoolExecutor (not multiprocessing.Pool): avoids pickling ~6 MB
      images through pipes on Windows and repeated process spawning.
    - grayscale=True in convert_from_path: 1/3 memory, native pdftoppm flag.
    - Text cache in a separate .ocr_cache.json file, NOT in the manifest:
      keeps the manifest tiny (~page-number list only).
    - Searchable PDF built at the end in one pass using Tesseract's native
      PDF output (image_to_pdf_or_hocr) which produces properly positioned
      invisible text, or skipped entirely when GENERATE_SEARCHABLE_PDF is False.
    - Single ThreadPoolExecutor across all batches (no per-batch spawn).
    - Per-page tqdm progress bar.
    """
    from pdf2image import convert_from_path
    from io import BytesIO

    doc_info = fitz.open(pdf_path)
    total_pages = len(doc_info)
    doc_info.close()

    # --- Resume support ---
    entry = manifest.get(manifest_key, {})
    completed_pages = set(entry.get("completed_pages", []))

    # Text is cached in a separate file (keeps manifest small)
    text_cache_path = pdf_path.with_suffix(".ocr_cache.json")
    if completed_pages and text_cache_path.exists():
        with open(text_cache_path, "r", encoding="utf-8") as f:
            all_text = {int(k): v for k, v in json.load(f).items()}
    else:
        all_text = {}
        completed_pages = set()

    # --- Per-page progress bar ---
    pbar = tqdm(
        total=total_pages, initial=len(completed_pages),
        desc=f"  OCR {pdf_path.stem}", unit="pg", leave=False,
    )

    # --- OCR in batches with a single persistent thread pool ---
    with ThreadPoolExecutor(max_workers=OCR_WORKERS) as executor:
        for batch_start in range(0, total_pages, OCR_BATCH_SIZE):
            batch_end = min(batch_start + OCR_BATCH_SIZE, total_pages)
            batch_pages = list(range(batch_start, batch_end))

            # Skip fully-completed batches
            if all(p in completed_pages for p in batch_pages):
                continue

            # Convert this batch — grayscale natively via pdftoppm (-gray)
            images = convert_from_path(
                pdf_path,
                dpi=OCR_DPI,
                first_page=batch_start + 1,
                last_page=batch_end,
                grayscale=True,
            )

            # Submit only pages not yet completed
            futures = {}
            for i, page_num in enumerate(batch_pages):
                if page_num not in completed_pages:
                    future = executor.submit(_ocr_single_page, page_num, images[i])
                    futures[future] = page_num

            # Collect results as they finish (updates progress per page)
            for future in as_completed(futures):
                page_num, text = future.result()
                all_text[page_num] = f"--- Page {page_num + 1} ---\n{text}"
                completed_pages.add(page_num)
                pbar.update(1)

            del images

            # Checkpoint: text → separate cache file, manifest stays small
            with open(text_cache_path, "w", encoding="utf-8") as f:
                json.dump({str(k): v for k, v in all_text.items()}, f)

            manifest[manifest_key] = {
                "status": "in_progress",
                "page_count": total_pages,
                "completed_pages": sorted(completed_pages),
                "last_checkpoint": datetime.now().isoformat(),
            }
            save_manifest(manifest)
            logger.info(
                f"{manifest_key}: Batch pages {batch_start + 1}-{batch_end} done "
                f"({len(completed_pages)}/{total_pages})"
            )

    pbar.close()

    # --- Optional: build searchable PDF using Tesseract's native PDF output ---
    # image_to_pdf_or_hocr produces a PDF page with the image as background
    # and properly positioned invisible text overlay — making it truly searchable.
    if GENERATE_SEARCHABLE_PDF:
        import pytesseract

        logger.info(f"{manifest_key}: Building searchable PDF ({total_pages} pages)...")
        searchable_doc = fitz.open()
        for batch_start in tqdm(
            range(0, total_pages, OCR_BATCH_SIZE),
            desc=f"  Searchable PDF {pdf_path.stem}",
            unit="batch", leave=False,
        ):
            batch_end = min(batch_start + OCR_BATCH_SIZE, total_pages)
            images = convert_from_path(
                pdf_path, dpi=OCR_DPI,
                first_page=batch_start + 1, last_page=batch_end,
                grayscale=True,
            )
            for image in images:
                pdf_bytes = pytesseract.image_to_pdf_or_hocr(image, extension="pdf")
                page_doc = fitz.open("pdf", pdf_bytes)
                searchable_doc.insert_pdf(page_doc)
                page_doc.close()
            del images
        searchable_doc.save(output_searchable_path)
        searchable_doc.close()
        logger.info(f"{manifest_key}: Searchable PDF saved.")

    # Clean up cache file on successful completion
    if text_cache_path.exists():
        text_cache_path.unlink()

    # Return full text in page order
    return "\n\n".join(all_text[p] for p in range(total_pages) if p in all_text)

## Process PDFs

In [ ]:
processed = 0
skipped = 0
ocr_count = 0
text_extract_count = 0
errors = 0

for pdf_path in tqdm(pdf_files, desc="Processing PDFs"):
    filename = pdf_path.name
    txt_path = pdf_path.with_suffix(".txt")
    searchable_path = pdf_path.with_name(pdf_path.stem + "_searchable.pdf")

    # Skip if already fully processed (but allow resuming "in_progress")
    entry = manifest.get(filename, {})
    if entry.get("status") == "done" and txt_path.exists():
        skipped += 1
        continue

    try:
        doc = fitz.open(pdf_path)
        page_count = len(doc)
        doc.close()

        searchable = is_searchable(pdf_path)

        if searchable:
            logger.info(f"{filename}: Searchable PDF, extracting text directly")
            text = extract_text_pymupdf(pdf_path)
            is_ocr = False
            text_extract_count += 1
        else:
            status = entry.get("status", "")
            done_pages = len(entry.get("completed_pages", []))
            if status == "in_progress":
                logger.info(
                    f"{filename}: Resuming OCR from page {done_pages + 1}/{page_count}"
                )
            else:
                logger.info(
                    f"{filename}: Scanned PDF, running OCR ({page_count} pages)"
                )
            text = ocr_pdf(pdf_path, searchable_path, manifest, filename)
            is_ocr = True
            ocr_count += 1

        # Save extracted text
        with open(txt_path, "w", encoding="utf-8") as f:
            f.write(text)

        # Update manifest — mark fully done, drop checkpoint fields
        manifest[filename] = {
            "status": "done",
            "is_ocr": is_ocr,
            "page_count": page_count,
            "text_file": txt_path.name,
            "searchable_pdf": searchable_path.name if is_ocr else None,
            "processed_at": datetime.now().isoformat(),
        }
        save_manifest(manifest)
        processed += 1

        logger.info(f"{filename}: Done — {page_count} pages, OCR={is_ocr}")

    except Exception as e:
        logger.error(f"{filename}: Processing failed — {e}")
        manifest[filename] = {
            **manifest.get(filename, {}),
            "status": "error",
            "error": str(e),
            "processed_at": datetime.now().isoformat(),
        }
        save_manifest(manifest)
        errors += 1

Processing PDFs:   0%|          | 0/860 [00:00<?, ?it/s]

[INFO] Volume_135.pdf: Resuming OCR from page 401/634


  OCR Volume_135:  63%|######3   | 400/634 [00:00<?, ?pg/s]

[INFO] Volume_135.pdf: Batch pages 401-450 done (450/634)
[INFO] Volume_135.pdf: Batch pages 451-500 done (500/634)
[INFO] Volume_135.pdf: Batch pages 501-550 done (550/634)
[INFO] Volume_135.pdf: Batch pages 551-600 done (600/634)
[INFO] Volume_135.pdf: Batch pages 601-634 done (634/634)
[INFO] Volume_135.pdf: Done — 634 pages, OCR=True
[INFO] Volume_136.pdf: Scanned PDF, running OCR (681 pages)


  OCR Volume_136:   0%|          | 0/681 [00:00<?, ?pg/s]

...

[INFO] Volume_192.pdf: Batch pages 1-50 done (50/853)
[INFO] Volume_192.pdf: Batch pages 51-100 done (100/853)
[INFO] Volume_192.pdf: Batch pages 101-150 done (150/853)
[INFO] Volume_192.pdf: Batch pages 151-200 done (200/853)
[INFO] Volume_192.pdf: Batch pages 201-250 done (250/853)
[INFO] Volume_192.pdf: Batch pages 251-300 done (300/853)
[INFO] Volume_192.pdf: Batch pages 301-350 done (350/853)
[INFO] Volume_192.pdf: Batch pages 351-400 done (400/853)
[INFO] Volume_192.pdf: Batch pages 401-450 done (450/853)
[INFO] Volume_192.pdf: Batch pages 451-500 done (500/853)
[INFO] Volume_192.pdf: Batch pages 501-550 done (550/853)
[INFO] Volume_192.pdf: Batch pages 551-600 done (600/853)
[INFO] Volume_192.pdf: Batch pages 601-650 done (650/853)
[INFO] Volume_192.pdf: Batch pages 651-700 done (700/853)
[INFO] Volume_192.pdf: Batch pages 701-750 done (750/853)
[INFO] Volume_192.pdf: Batch pages 751-800 done (800/853)
[INFO] Volume_192.pdf: Batch pages 801-850 done (850/853)
[INFO] Volume_192.p

  OCR Volume_193:   0%|          | 0/871 [00:00<?, ?pg/s]

[INFO] Volume_193.pdf: Batch pages 1-50 done (50/871)
[INFO] Volume_193.pdf: Batch pages 51-100 done (100/871)
[INFO] Volume_193.pdf: Batch pages 101-150 done (150/871)
[INFO] Volume_193.pdf: Batch pages 151-200 done (200/871)
[INFO] Volume_193.pdf: Batch pages 201-250 done (250/871)
[INFO] Volume_193.pdf: Batch pages 251-300 done (300/871)
[INFO] Volume_193.pdf: Batch pages 301-350 done (350/871)
[INFO] Volume_193.pdf: Batch pages 351-400 done (400/871)
[INFO] Volume_193.pdf: Batch pages 401-450 done (450/871)
[INFO] Volume_193.pdf: Batch pages 451-500 done (500/871)
[INFO] Volume_193.pdf: Batch pages 501-550 done (550/871)
[INFO] Volume_193.pdf: Batch pages 551-600 done (600/871)
[INFO] Volume_193.pdf: Batch pages 601-650 done (650/871)
[INFO] Volume_193.pdf: Batch pages 651-700 done (700/871)
[INFO] Volume_193.pdf: Batch pages 701-750 done (750/871)
[INFO] Volume_193.pdf: Batch pages 751-800 done (800/871)
[INFO] Volume_193.pdf: Batch pages 801-850 done (850/871)
[INFO] Volume_193.p

  OCR Volume_194:   0%|          | 0/767 [00:00<?, ?pg/s]

[INFO] Volume_194.pdf: Batch pages 1-50 done (50/767)
[INFO] Volume_194.pdf: Batch pages 51-100 done (100/767)
[INFO] Volume_194.pdf: Batch pages 101-150 done (150/767)
[INFO] Volume_194.pdf: Batch pages 151-200 done (200/767)
[INFO] Volume_194.pdf: Batch pages 201-250 done (250/767)
[INFO] Volume_194.pdf: Batch pages 251-300 done (300/767)
[INFO] Volume_194.pdf: Batch pages 301-350 done (350/767)


## Summary

In [ ]:
summary = (
    f"\n{'='*40}\n"
    f"Processing Summary\n"
    f"{'='*40}\n"
    f"Total PDFs found:      {len(pdf_files)}\n"
    f"Processed this run:    {processed}\n"
    f"  - Text extraction:   {text_extract_count}\n"
    f"  - OCR:               {ocr_count}\n"
    f"Skipped (already done):{skipped}\n"
    f"Errors:                {errors}\n"
    f"{'='*40}"
)
logger.info(summary)
print(summary)

[INFO] 
Processing Summary
Total PDFs found:      860
Processed this run:    483
  - Text extraction:   483
  - OCR:               0
Skipped (already done):0
Errors:                377



Processing Summary
Total PDFs found:      860
Processed this run:    483
  - Text extraction:   483
  - OCR:               0
Skipped (already done):0
Errors:                377
